# Atom-level knowledge graph

This notebook builds and explores the atom-level knowledge graph for the WGBO
corpus and its Book 6 fallbacks. Each Dutch Civil Code article is decomposed
into atoms defined as the smallest units carrying a single legal meaning, annotated
against a 9-label schema, and connected into a graph through four edge
channels: shared annotation tags (IDF-weighted), semantic similarity between
tag values, hierarchical containment through the Dutch legal tree, and
explicit cross-article references.

On top of that graph, a classifier ranks atoms against a query, and a
chained-reasoning walker expands from the top matches through the graph so
every result names the edge that surfaced it.

Section flow:

1. Load the atom table
2. Corpus overview
3. Cross-article edges
4. Article-level connectivity
5. Tag weighting (IDF)
6. Semantic similarity
7. Atom classifier and evaluation
8. Legal hierarchy : placing atoms in the Dutch civil code tree
9. Chained reasoning demo : walking the graph from a query

Model comparison and alpha/top-k sweeps live in `02_experiments.ipynb`.

## 1. Load the atom table


In [1]:
import sys
import os
from pathlib import Path


def find_project_root(marker='src/data/atom_table.py'):
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError(f"Could not find project root (no {marker} in any parent)")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 220)

from src.data.atom_table import (
    build_atom_table, summary_stats, tag_frequencies, show_atom,
    build_edge_table, add_edge_summary, article_connectivity,
    compute_tag_weights, show_tag_weights,
    compute_tag_similarities, build_semantic_edge_table, unique_tags_by_field,
    classify_atom, evaluate_classifier_loo,
    LIST_FIELDS,
)

df = build_atom_table()
print(f'Loaded {len(df)} atoms across {df["article_id"].nunique()} articles')
df.head()


Loaded 82 atoms across 26 articles


,atom_id,article_id,lid,condition_type,text,text_nl,actors,legal_relations,acts,geographical_domain,temporal,explicit_references,hierarchies,residual,source,annotator,annotated_at,notes
0,6:162(1).a,6:162,1,post_condition,"One who commits an unlawful act against another, which can be attr...","Hij die jegens een ander een onrechtmatige daad pleegt, welke hem ...","[wrongdoer, injured party]",[],"[commits, compensates]",[],[],[],[],"[unlawful act, damage, attribution]","Dutch Civil Code, Book 6, Title 6.3.1",Claude-draft,2026-07-01,Central tort rule — parallel to 6:74's central contract rule. Dama...
1,6:162(2).a,6:162,2,post_condition,An unlawful act is deemed to be an infringement of a right and an ...,Als onrechtmatige daad worden aangemerkt een inbreuk op een recht ...,[],[],[],[],[],[],[],"[unlawful act, infringement, right, act, omission, legal duty, unw...","Dutch Civil Code, Book 6, Title 6.3.1",Claude-draft,2026-07-01,Definitional atom for 'unlawful act'. AND-connected clauses within...
2,6:162(2).b,6:162,2,pre_condition,all this subject to the presence of a ground for justification,een en ander behoudens de aanwezigheid van een rechtvaardigingsgrond,[],[],[],[],[],[],[subject to the presence of a ground for justification],"[justification, ground]","Dutch Civil Code, Book 6, Title 6.3.1",Claude-draft,2026-07-01,Hierarchy tag captures 'subject to' — parallels 7:454(3).a treatme...
3,6:162(3).a,6:162,3,post_condition,An unlawful act can be attributed to the perpetrator if it is due ...,"Een onrechtmatige daad kan aan de dader worden toegerekend, indien...",[wrongdoer],[],[],[],[],[],[],"[unlawful act, attribution, fault, law, generally accepted views, ...","Dutch Civil Code, Book 6, Title 6.3.1",Claude-draft,2026-07-01,Attribution rule for tort. Note shared residuals with 6:75 ('fault...
4,6:170(1).a,6:170,1,post_condition,"For damage caused to a third party by a fault of a subordinate, th...","Voor schade, aan een derde toegebracht door een fout van een onder...","[employer, subordinate, third party]",[service relationship],[],[],[],[],[],"[damage, fault, liability, task]","Dutch Civil Code, Book 6, Title 6.3.2",Claude-draft,2026-07-01,Tort-based vicarious liability. Employer/subordinate/third party i...


## 2. Corpus overview

Per-article summary followed by tag frequencies for each schema field.

Lists all articles and their attributes (number of : atoms, post-conditions, pre-conditions, legal relations, acts, geographical domain, temporal, explicit references, hierarchies, residual).


In [2]:
summary_stats(df)


,article_id,n_atoms,n_pre,n_post,actors,legal_relations,acts,geographical_domain,temporal,explicit_references,hierarchies,residual
0,6:162,4,1,3,2,0,1,0,0,0,1,4
1,6:170,3,2,1,3,2,0,0,0,0,0,3
2,6:74,3,1,2,3,1,1,0,0,1,0,3
3,6:75,1,0,1,1,0,0,0,0,0,0,1
4,6:76,1,0,1,1,1,1,0,0,0,0,1
5,7:446,5,1,4,4,2,1,0,0,1,0,5
6,7:448,8,2,6,8,4,8,0,2,1,0,8
7,7:453,2,0,2,2,1,2,0,0,0,0,2
8,7:454,8,2,6,7,2,6,0,3,1,1,8
9,7:455,3,2,1,2,1,1,0,1,1,2,3


In [3]:
for field in LIST_FIELDS:
    freqs = tag_frequencies(df, field)
    if len(freqs) == 0:
        continue
    print(f'\n--- {field} ---')
    print(freqs.to_string())



--- actors ---
actors
patient                                   30
care provider                             21
debtor                                     5
someone other than patient                 5
colonist                                   5
employer                                   3
subordinate                                3
citizen                                    3
wrongdoer                                  2
client                                     2
unemancipated minor                        2
injured party                              1
third party                                1
creditor                                   1
auxiliary person                           1
physician                                  1
dentist                                    1
substitute                                 1
directly involved persons                  1
attending physician                        1
certified emergency medical technician     1
orbital health authority        

## 3. Cross-article edges

One row per pair of atoms from different articles that share at least one tag
value in the same field. Each edge is one overlapping tag. Baseline strength
is the count of shared tags in that field.

Overview of 'Exact Match' edges, edges that are formed purely through a shared value within a field (field being actors, legal relations, residuals etc...)


In [4]:
edges = build_edge_table(df)
print(f'{len(edges)} cross-article edges')
print()
print('Edges by field:')
print(edges['field'].value_counts().to_string())
edges


639 cross-article edges

Edges by field:
field
actors                 498
legal_relations         68
residual                55
acts                     9
explicit_references      6
temporal                 3


,atom_a,atom_b,article_a,article_b,field,shared,strength,weighted_strength
0,6:162(1).a,6:170(1).a,6:162,6:170,residual,[damage],1,1.0
1,6:162(1).a,6:74(1).a,6:162,6:74,acts,[compensates],1,1.0
2,6:162(1).a,6:74(1).a,6:162,6:74,residual,[damage],1,1.0
3,6:162(1).a,6:74(1).b,6:162,6:74,residual,[attribution],1,1.0
4,6:162(1).a,6:75(1).a,6:162,6:75,residual,[attribution],1,1.0
...,...,...,...,...,...,...,...,...
634,synth:11(1).a,synth:13(1).c,synth:11,synth:13,actors,[patient],1,1.0
635,synth:11(1).b,synth:13(1).c,synth:11,synth:13,actors,[patient],1,1.0
636,synth:11(1).c,synth:13(1).c,synth:11,synth:13,actors,[patient],1,1.0
637,synth:13(1).a,synth:15(1).a,synth:13,synth:15,actors,[colonist],1,1.0


### Atoms as hubs and isolates

For each atom, the number of edges it has and which schema fields produce
them. High `n_edges` = hub. Zero edges = the atom shares no tags with any
other article's atoms.


In [5]:
df_e = add_edge_summary(df, edges)
df_e.sort_values('n_edges', ascending=False)[
    ['atom_id', 'n_edges', 'connected_atoms', 'edge_fields']
]


,atom_id,n_edges,connected_atoms,edge_fields
35,7:455(1).a,52,"[7:446(1).a, 7:446(3).a, 7:446(5).a, 7:448(1).a, 7:448(1).b, 7:448...","[actors, legal_relations, residual]"
38,7:457(1).a,48,"[7:446(1).a, 7:446(3).a, 7:446(5).a, 7:448(1).a, 7:448(1).b, 7:448...","[actors, acts, legal_relations, residual]"
41,7:457(2).a,47,"[6:74(1).a, 6:74(2).a, 6:76(1).a, 7:446(1).a, 7:446(3).a, 7:446(5)...","[actors, acts, legal_relations, residual]"
12,7:446(1).a,44,"[7:448(1).a, 7:448(1).b, 7:448(1).c, 7:448(2).a, 7:448(3).a, 7:448...","[actors, legal_relations]"
32,7:454(2).a,43,"[7:446(1).a, 7:446(3).a, 7:446(5).a, 7:448(1).a, 7:448(1).b, 7:448...","[actors, legal_relations, residual, temporal]"
...,...,...,...,...
68,synth:10(1).c,0,[],[]
77,synth:14(1).a,0,[],[]
75,synth:13(1).b,0,[],[]
78,synth:14(1).b,0,[],[]


In [6]:
df_e.loc[df_e['n_edges'] == 0, ['atom_id', 'condition_type', 'text']]


,atom_id,condition_type,text
1,6:162(2).a,post_condition,An unlawful act is deemed to be an infringement of a right and an ...
2,6:162(2).b,pre_condition,all this subject to the presence of a ground for justification
13,7:446(2).a,post_condition,Actions in the field of medicine include all procedures directly r...
14,7:446(2).b,post_condition,Actions in the field of medicine also include other actions direct...
46,synth:2(1).a,pre_condition,If a medical emergency occurs during an extravehicular surface tra...
47,synth:2(1).b,post_condition,any certified emergency medical technician may administer autonomo...
49,synth:3(1).b,post_condition,the orbital health authority must purge the specified data from al...
50,synth:4(1).a,pre_condition,If a pathogenic mutation is detected within a hydroponic agricultu...
52,synth:5(1).a,pre_condition,If an unemancipated minor undergoes artificial gravity acclimation...
56,synth:6(1).c,pre_condition,unless the pathogen is legally classified as a colony-threatening ...


## 4. Article-level connectivity

Aggregate atom-level edges up to article pairs. Useful for a first look at
which articles cluster together at the coarse level.

Overview of total edges across different *articles* and their shared fields.


In [7]:
article_connectivity(edges)


,article_a,article_b,n_edges,fields
0,6:162,6:170,4,[residual]
1,6:162,6:74,4,"[acts, residual]"
2,6:162,6:75,2,[residual]
3,6:162,7:455,1,[residual]
4,6:162,7:457,1,[residual]
...,...,...,...,...
73,synth:8,synth:13,3,[actors]
74,synth:8,synth:15,2,[actors]
75,synth:8,synth:9,1,[actors]
76,synth:9,synth:13,1,[actors]


## 5. Tag weighting

Common tags (`patient`, `care provider`) appear in most atoms and carry
little discriminative signal. IDF down-weights them and up-weights rare
tags. Two modes: `idf` (default, log inverse document frequency) and `tf`
(raw frequency).

Lists bottom 8 tags ( words or phrases that are very common they appear frequently across corpus so does not help in inferring knowledge).

Second output lists edges with the highest weights ie edges formed across tags that are lesser common accross the corpus.


In [8]:
weights_idf = compute_tag_weights(df, method='idf')
print(f'Computed {len(weights_idf)} tag weights')
print()
print('Top 8 (most discriminative):')
print(show_tag_weights(weights_idf).head(8).to_string())
print()
print('Bottom 8 (least discriminative):')
print(show_tag_weights(weights_idf).tail(8).to_string())


Computed 312 tag weights

Top 8 (most discriminative):
      field                                             tag  weight
0  residual                  guaranteed mechanical mobility  3.7257
1    actors                                   injured party  3.7257
2  residual            internal biological integration risk  3.7257
3    actors                                     third party  3.7257
4  residual                                       procedure  3.7257
5  residual                                cross-referenced  3.7257
6  residual  unclassified extraterrestrial biological agent  3.7257
7  residual                       compromised network nodes  3.7257

Bottom 8 (least discriminative):
               field                  tag  weight
304         residual                fault  2.6271
305           actors             colonist  2.6271
306         residual                 file  2.4729
307         residual          information  2.4729
308             acts             provides  2.4729

In [9]:
edges_weighted = build_edge_table(df, weights=weights_idf)

print('Top 5 by uniform strength (count of shared tags):')
print(edges_weighted.sort_values('strength', ascending=False).head(5)[
    ['atom_a', 'atom_b', 'shared', 'strength']
].to_string(index=False))
print()
print('Top 5 by IDF-weighted strength:')
print(edges_weighted.sort_values('weighted_strength', ascending=False).head(5)[
    ['atom_a', 'atom_b', 'shared', 'weighted_strength']
].to_string(index=False))


Top 5 by uniform strength (count of shared tags):
    atom_a     atom_b                                                        shared  strength
6:162(3).a  6:75(1).a [attribution, commerce, fault, generally accepted views, law]         5
7:448(3).b 7:457(1).a          [care provider, patient, someone other than patient]         3
7:455(2).a 7:457(1).a                         [patient, someone other than patient]         2
7:455(1).a 7:457(1).a                                      [care provider, patient]         2
7:454(2).a 7:457(2).a                                      [care provider, patient]         2

Top 5 by IDF-weighted strength:
    atom_a     atom_b                                                        shared  weighted_strength
6:162(3).a  6:75(1).a [attribution, commerce, fault, generally accepted views, law]            14.8863
 6:74(1).b  6:75(1).a                             [attribution, failure to perform]             5.8419
7:454(2).a 7:455(1).a                       

## 6. Semantic similarity

Exact set intersection misses synonyms and related concepts. The semantic
layer computes cosine similarity between every pair of unique tag values in
each field, using a sentence-transformer. Pairs above the threshold become
implicit tag equivalences, which then promote to atom-level edges.

Expanding on exact matches by adding semantic similarity to form edges across field values that are also similar enough to form an edge, 'enough' determined by threshold currently set to 0.55.
Still Unsure what model works best LegalBert did not preform as well as I expected so switched back to mpnet for now.

Lists similarity pairs and number of semantic edges


In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2') #not sure which to use still
sim_pairs = compute_tag_similarities(df, threshold=0.55, model=model)
sem_edges = build_semantic_edge_table(df, sim_pairs)

print(f'{len(sim_pairs)} high-similarity tag pairs')
print(f'{len(sem_edges)} semantic edges across atoms')
sim_pairs.head(10)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

114 high-similarity tag pairs
374 semantic edges across atoms


,field,tag_a,tag_b,similarity
0,explicit_references,article 454,article 455,0.9806
1,explicit_references,paragraph 1,paragraph 2,0.9579
2,residual,procedure,procedures,0.9258
3,explicit_references,article 448,article 454,0.9088
4,explicit_references,article 455,article 465,0.9082
5,explicit_references,article 454,article 465,0.9042
6,residual,obligation,obligations,0.8974
7,actors,creditor,debtor,0.8954
8,explicit_references,article 448,article 455,0.8725
9,explicit_references,article 450,article 454,0.8694


### Exact vs semantic coverage

Both channels operate on the same atoms. Exact edges are deterministic and
auditable; semantic edges catch synonyms exact matching misses.

Summary/Comparison of Semantic and Exact Edges.


In [11]:
# Combine both channels in one table
exact = edges.copy()
exact['edge_kind'] = 'exact'

sem = sem_edges.copy()
sem['edge_kind'] = 'semantic'
sem['shared'] = sem['matched_pairs']
sem['strength'] = sem['n_matches']
sem['weighted_strength'] = sem['max_similarity']

cols = ['edge_kind', 'atom_a', 'atom_b', 'field', 'shared', 'strength', 'weighted_strength']
combined = pd.concat([exact[cols], sem[cols]], ignore_index=True)

# Coverage overlap
exact_pairs = {(r.atom_a, r.atom_b) for _, r in exact.iterrows()}
sem_pairs_set = {(r.atom_a, r.atom_b) for _, r in sem.iterrows()}
sem_only = sem_pairs_set - exact_pairs
exact_only = exact_pairs - sem_pairs_set
both = sem_pairs_set & exact_pairs

print(f'Exact edges:                {len(exact)}')
print(f'Semantic edges:             {len(sem_edges)}')
print(f'Atom pairs, exact only:     {len(exact_only)}')
print(f'Atom pairs, semantic only:  {len(sem_only)}')
print(f'Atom pairs in both:         {len(both)}')

if sem_only:
    print()
    print('Atom pairs connected ONLY semantically (no exact tag overlap):')
    for a, b in sorted(sem_only):
        print(f'  {a} <-> {b}')


Exact edges:                639
Semantic edges:             374
Atom pairs, exact only:     365
Atom pairs, semantic only:  162
Atom pairs in both:         173

Atom pairs connected ONLY semantically (no exact tag overlap):
  6:162(1).a <-> 7:448(3).a
  6:162(1).a <-> 7:454(3).a
  6:162(1).a <-> 7:455(2).b
  6:162(1).a <-> 7:457(1).b
  6:162(2).a <-> 6:75(1).a
  6:162(2).a <-> 7:446(5).a
  6:162(2).a <-> 7:454(3).b
  6:162(2).a <-> 7:455(2).b
  6:162(2).a <-> 7:457(1).c
  6:162(3).a <-> 7:446(1).a
  6:170(1).a <-> 7:446(5).a
  6:170(1).a <-> 7:448(3).a
  6:170(1).a <-> 7:453(1).b
  6:170(1).a <-> 7:455(2).b
  6:170(1).a <-> 7:457(1).b
  6:74(1).a <-> 7:448(3).a
  6:74(1).a <-> 7:455(2).b
  6:74(1).a <-> 7:457(1).b
  6:74(2).a <-> 7:457(3).b
  6:75(1).a <-> 7:446(1).a
  6:76(1).a <-> 7:446(5).a
  6:76(1).a <-> 7:453(1).b
  7:446(1).a <-> 7:457(1).b
  7:446(1).a <-> synth:14(1).a
  7:446(1).a <-> synth:15(1).b
  7:446(1).a <-> synth:2(1).a
  7:446(2).a <-> 7:448(1).a
  7:446(2).a <-> 7:4

## 7. Atom classifier

Given an atom-shaped input, rank all corpus atoms by similarity and predict
an article. Score composition:

- **exact_score** : sum of IDF weights of tags shared with the query
- **semantic_score** : sum of best semantic-similarity matches for tags not exactly matched
- **combined_score** = `alpha * exact_score + (1 - alpha) * semantic_score`

`alpha=0.5` weights both channels equally.


### 7.1 Classify an existing atom

Outputs Justification for classification.


In [12]:
query = df[df['atom_id'] == '7:454(2).a'].iloc[0].to_dict()
result = classify_atom(query, df, weights=weights_idf, sim_pairs=sim_pairs, alpha=0.5, top_k=5)
print(f"Predicted article: {result['predicted_article']}")
result['top_matches']


Predicted article: 7:454


,atom_id,article_id,combined_score,exact_score,semantic_score
0,7:455(1).a,7:455,5.1611,9.5979,0.7242
1,7:448(1).b,7:448,3.7063,7.4127,0.0000
2,7:454(1).a,7:454,3.2827,6.5654,0.0000
3,7:454(1).b,7:454,2.3928,4.7856,0.0000
4,7:455(2).a,7:455,2.3080,4.0174,0.5987


### 7.2 Leave-one-out (LOO) evaluation

For each atom: remove it, classify against the rest, record whether the
correct article appears at rank 1 and in top-k, plus reciprocal rank.


In [13]:
eval_df, summary = evaluate_classifier_loo(df, weights=weights_idf, sim_pairs=sim_pairs, alpha=0.5, top_k=3)
print('Overall:')
for k, v in summary.items():
    print(f'  {k:25s} {v}')
print()
eval_df[['atom_id', 'true_article', 'top1_match', 'top1_hit', 'rank']]


Overall:
  n_atoms                   82
  top1_accuracy             0.378
  top3_accuracy             0.5
  mean_reciprocal_rank      0.4309



,atom_id,true_article,top1_match,top1_hit,rank
0,6:162(1).a,6:162,6:162,True,1.0
1,6:162(2).a,6:162,6:162,True,1.0
2,6:162(2).b,6:162,6:162,True,1.0
3,6:162(3).a,6:162,6:75,False,2.0
4,6:170(1).a,6:170,6:170,True,1.0
...,...,...,...,...,...
77,synth:14(1).a,synth:14,synth:2,False,NaN
78,synth:14(1).b,synth:14,synth:13,False,NaN
79,synth:15(1).a,synth:15,synth:3,False,NaN
80,synth:15(1).b,synth:15,synth:15,True,1.0


### 7.3 Per-article breakdown


In [14]:
per_article = eval_df.groupby('true_article').agg(
    n_atoms=('atom_id', 'size'),
    top1_accuracy=('top1_hit', 'mean'),
    topk_accuracy=('topk_hit', 'mean'),
    mrr=('reciprocal_rank', 'mean'),
).round(3)
per_article


,n_atoms,top1_accuracy,topk_accuracy,mrr
true_article,,,,
6:162,4,0.750,1.000,0.875
6:170,3,1.000,1.000,1.000
6:74,3,0.333,1.000,0.611
6:75,1,0.000,0.000,0.000
6:76,1,0.000,0.000,0.000
7:446,5,1.000,1.000,1.000
7:448,8,0.750,0.875,0.792
7:453,2,0.000,0.000,0.000
7:454,8,0.375,0.750,0.521


### 7.4 Misclassifications

Atoms where the correct article didn't reach rank 1. Errors that look like
sibling articles are usually intuitive; errors that jump across topics are
worth digging into.


In [15]:
misses = eval_df[~eval_df['top1_hit']]
misses[['atom_id', 'true_article', 'top1_match', 'rank']]


,atom_id,true_article,top1_match,rank
3,6:162(3).a,6:162,6:75,2.0
7,6:74(1).a,6:74,6:76,3.0
8,6:74(1).b,6:74,6:75,2.0
10,6:75(1).a,6:75,6:162,NaN
11,6:76(1).a,6:76,6:74,NaN
18,7:448(1).b,7:448,7:455,3.0
22,7:448(3).b,7:448,7:457,NaN
25,7:453(1).a,7:453,7:448,NaN
26,7:453(1).b,7:453,7:446,NaN
27,7:454(1).a,7:454,7:448,3.0


### 7.5 Filter before semantic retrieval

A structural pre-filter before the classifier scores anything. The filter narrows the candidate set to atoms that share at least one tag with the query on the specified fields (default: `actors` and `acts`). The classifier then scores only the survivors.

At the current corpus size this is architecturally interesting rather than performance-critical — the classifier already isolates the right answers on 61 atoms without pre-filtering (as per 04/09/2026). The value shows at scale, and it makes the retrieval pipeline structurally.

In [16]:
from src.data.atom_table import filter_candidates

# Same LOO query as 7.2, run twice: once with the filter, once without
query = df[df['atom_id'] == '7:454(2).a'].iloc[0].to_dict()

result_no_filter = classify_atom(
    query, df.drop(df[df['atom_id'] == '7:454(2).a'].index),
    weights=weights_idf, sim_pairs=sim_pairs, alpha=0.5, top_k=5,
)

result_with_filter = classify_atom(
    query, df.drop(df[df['atom_id'] == '7:454(2).a'].index),
    weights=weights_idf, sim_pairs=sim_pairs, alpha=0.5, top_k=5,
    filter_fields=['actors', 'acts'],
)

print(f'Without filter: {len(result_no_filter["scores_by_atom"])} candidates scored')
print(f'With filter:    {len(result_with_filter["scores_by_atom"])} candidates scored')
print()
print('Top-5 without filter:')
print(result_no_filter['top_matches'].to_string(index=False))
print()
print('Top-5 with filter:')
print(result_with_filter['top_matches'].to_string(index=False))

Without filter: 81 candidates scored
With filter:    35 candidates scored

Top-5 without filter:
   atom_id article_id  combined_score  exact_score  semantic_score
7:455(1).a      7:455          5.1611       9.5979          0.7242
7:448(1).b      7:448          3.7063       7.4127          0.0000
7:454(1).a      7:454          3.2827       6.5654          0.0000
7:454(1).b      7:454          2.3928       4.7856          0.0000
7:455(2).a      7:455          2.3080       4.0174          0.5987

Top-5 with filter:
   atom_id article_id  combined_score  exact_score  semantic_score
7:455(1).a      7:455          5.1611       9.5979          0.7242
7:448(1).b      7:448          3.7063       7.4127          0.0000
7:454(1).a      7:454          3.2827       6.5654          0.0000
7:454(1).b      7:454          2.3928       4.7856          0.0000
7:455(2).a      7:455          2.3080       4.0174          0.5987


### 7.6 LOO comparison : filtered vs unfiltered

Full leave-one-out evaluation with and without the filter. Compares
top-1 accuracy, top-3 accuracy, and MRR side by side.

In [17]:
_, summary_no_filter = evaluate_classifier_loo(
    df, weights=weights_idf, sim_pairs=sim_pairs, alpha=0.5, top_k=3,
)
print('Baseline (no filter):')
for k, v in summary_no_filter.items():
    print(f'  {k:25s} {v}')
print()

# The evaluate_classifier_loo helper doesn't accept filter_fields, so running
# a hand-rolled LOO here to keep the change surgical.
from src.data.atom_table import classify_atom as _classify

filtered_hits = {'top1': 0, 'top3': 0, 'rr': []}
for idx, atom in df.iterrows():
    rest = df.drop(idx)
    res = _classify(atom.to_dict(), rest,
                    weights=weights_idf, sim_pairs=sim_pairs,
                    alpha=0.5, top_k=3, filter_fields=['actors', 'acts'])
    top = res['top_matches']
    if len(top) == 0:
        filtered_hits['rr'].append(0.0)
        continue
    hit_rank = None
    for r, art in enumerate(top['article_id'].values, start=1):
        if art == atom['article_id']:
            hit_rank = r
            break
    if hit_rank == 1: filtered_hits['top1'] += 1
    if hit_rank is not None: filtered_hits['top3'] += 1
    filtered_hits['rr'].append(1.0/hit_rank if hit_rank else 0.0)

n = len(df)
print('With filter (actors + acts):')
print(f"  top1_accuracy             {filtered_hits['top1']/n:.4f}")
print(f"  top3_accuracy             {filtered_hits['top3']/n:.4f}")
print(f"  mean_reciprocal_rank      {sum(filtered_hits['rr'])/n:.4f}")

Baseline (no filter):
  n_atoms                   82
  top1_accuracy             0.378
  top3_accuracy             0.5
  mean_reciprocal_rank      0.4309

No survivors found, falling back on full corpus
No survivors found, falling back on full corpus
No survivors found, falling back on full corpus
No survivors found, falling back on full corpus
No survivors found, falling back on full corpus
No survivors found, falling back on full corpus
No survivors found, falling back on full corpus
No survivors found, falling back on full corpus
No survivors found, falling back on full corpus
No survivors found, falling back on full corpus
No survivors found, falling back on full corpus
No survivors found, falling back on full corpus
No survivors found, falling back on full corpus
No survivors found, falling back on full corpus
With filter (actors + acts):
  top1_accuracy             0.3415
  top3_accuracy             0.5000
  mean_reciprocal_rank      0.4126


## 8. Legal hierarchy

Every article belongs somewhere in the Dutch civil code tree:

    Rechtsgebied  (private_law / public_law)
        Wetboek       (BW, Sr, Sv, ...)
            Boek          (1-10 for BW)
                Titel         (e.g. 7.7 - Opdracht)
                    Afdeling      (e.g. 7.7.5 - WGBO)
                        Artikel       (e.g. 7:454)

`src/data/hierarchy.py` encodes this tree and parses article IDs into their
taxonomy path. Book 7 is fully populated because that is where the WGBO
corpus lives. Other books have names but no title-level breakdown yet.


### 8.1 Parse taxonomy for the corpus

Overview of every atom's taxonomy (Which article, book, titel and afdeling it belongs to).


In [18]:
from src.data.hierarchy import (
    parse_article_id, add_taxonomy_columns, tree_summary,
    build_hierarchy_edges, build_reference_edges, evaluate_at_levels,
)

df_tax = add_taxonomy_columns(df)
df_tax[['atom_id', 'article_id', 'boek', 'titel', 'afdeling', 'taxonomy_path']].head(12)


,atom_id,article_id,boek,titel,afdeling,taxonomy_path
0,6:162(1).a,6:162,Boek 6,6.3,6.3.1,private_law > BW > Boek 6 > Titel 6.3 > Afd 6.3.1 > Art 6:162
1,6:162(2).a,6:162,Boek 6,6.3,6.3.1,private_law > BW > Boek 6 > Titel 6.3 > Afd 6.3.1 > Art 6:162
2,6:162(2).b,6:162,Boek 6,6.3,6.3.1,private_law > BW > Boek 6 > Titel 6.3 > Afd 6.3.1 > Art 6:162
3,6:162(3).a,6:162,Boek 6,6.3,6.3.1,private_law > BW > Boek 6 > Titel 6.3 > Afd 6.3.1 > Art 6:162
4,6:170(1).a,6:170,Boek 6,6.3,6.3.2,private_law > BW > Boek 6 > Titel 6.3 > Afd 6.3.2 > Art 6:170
5,6:170(1).b,6:170,Boek 6,6.3,6.3.2,private_law > BW > Boek 6 > Titel 6.3 > Afd 6.3.2 > Art 6:170
6,6:170(1).c,6:170,Boek 6,6.3,6.3.2,private_law > BW > Boek 6 > Titel 6.3 > Afd 6.3.2 > Art 6:170
7,6:74(1).a,6:74,Boek 6,6.1,6.1.9,private_law > BW > Boek 6 > Titel 6.1 > Afd 6.1.9 > Art 6:74
8,6:74(1).b,6:74,Boek 6,6.1,6.1.9,private_law > BW > Boek 6 > Titel 6.1 > Afd 6.1.9 > Art 6:74
9,6:74(2).a,6:74,Boek 6,6.1,6.1.9,private_law > BW > Boek 6 > Titel 6.1 > Afd 6.1.9 > Art 6:74


### 8.2 Tree summary
How many atoms formed at every level of the structure


In [19]:
tree_summary(df_tax)


,level,value,n_atoms
0,rechtsgebied,private_law,44
1,rechtsgebied,synthetic,38
2,wetboek,BW,44
3,wetboek,LMC,38
4,boek,(none),38
5,boek,Boek 7,32
6,boek,Boek 6,12
7,titel,(none),38
8,titel,7.7,32
9,titel,6.3,7


### 8.3 Containment edges

The hierarchy adds a new edge type  directed parent-child containment
edges from atoms up through the tree. These sit alongside the tag-overlap
and semantic edges from Sections 3 and 6.

So far all the edges are due to some similairty between atoms these new edges are different they mimic the structure of the Dutch Civil Code. They say "this thing is inside that thing." They're the edges of the tree structure itself.


In [20]:
hier_edges = build_hierarchy_edges(df_tax)
print(f'{len(hier_edges)} containment edges')
print()
print('Edges by level:')
print(hier_edges['level'].value_counts().to_string())


119 containment edges

Edges by level:
level
atom_to_article            82
article_to_wetboek         15
article_to_afdeling        11
afdeling_to_titel           4
titel_to_boek               3
boek_to_wetboek             2
wetboek_to_rechtsgebied     2


In [21]:
# Sample of the edges one row per unique containment relation
hier_edges.drop_duplicates(subset=['source', 'target'])


,source,target,edge_kind,level
0,6:162(1).a,6:162,contains,atom_to_article
1,6:162,6.3.1,contains,article_to_afdeling
2,6.3.1,6.3,contains,afdeling_to_titel
3,6.3,Boek 6,contains,titel_to_boek
4,Boek 6,BW,contains,boek_to_wetboek
...,...,...,...,...
114,synth:14(1).b,synth:14,contains,atom_to_article
115,synth:15(1).a,synth:15,contains,atom_to_article
116,synth:15,LMC,contains,article_to_wetboek
117,synth:15(1).b,synth:15,contains,atom_to_article


### 8.4 Cross-article reference edges

A second structural edge type, orthogonal to containment: directed edges from
an atom to any article it explicitly references. Read from the
`explicit_references` field on each atom.

Example: `7:454(3).a` contains "Without prejudice to the provisions of article
455" that produces a directed edge `7:454(3).a → 7:455`. These edges are
asymmetric on purpose: `7:455` does not reference `7:454` back.


In [22]:
ref_edges, ref_unresolved = build_reference_edges(df_tax, return_unresolved=True)
print(f'{len(ref_edges)} reference edges built')
print(f'{len(ref_unresolved)} reference strings unresolved (worth inspecting)')
ref_edges


5 reference edges built
0 reference strings unresolved (worth inspecting)


,source,target,edge_kind,ref_text
0,7:454(3).a,7:455,references,article 455
1,7:457(1).a,7:448,references,article 448
2,7:457(1).a,7:454,references,article 454
3,7:457(3).a,7:450,references,article 450
4,7:457(3).a,7:465,references,article 465


In [23]:
# Which articles are the busiest reference sources and targets?
if len(ref_edges):
    print('Sources (atoms that reference other articles):')
    print(ref_edges['source'].value_counts().to_string())
    print()
    print('Targets (articles being referenced):')
    print(ref_edges['target'].value_counts().to_string())
    print()
if len(ref_unresolved):
    print('Unresolved reference strings (need parser rules or annotation cleanup):')
    print(ref_unresolved.to_string(index=False))


Sources (atoms that reference other articles):
source
7:457(1).a    2
7:457(3).a    2
7:454(3).a    1

Targets (articles being referenced):
target
7:455    1
7:448    1
7:454    1
7:450    1
7:465    1



### 8.5 Hierarchical accuracy

Leave-one-out with accuracy reported at each level of the tree, not just at
the article level. For a corpus concentrated in one afdeling (WGBO), the
book/titel/afdeling accuracies are trivially high because there is little
variation to distinguish. This becomes meaningful once atoms from other
sections (e.g. Book 6 obligations) are added.


In [24]:
eval_hier, summary_hier = evaluate_at_levels(
    df_tax, weights=weights_idf, sim_pairs=sim_pairs, alpha=0.5, top_k=3
)
print('Accuracy by level (higher = coarser prediction):')
for k, v in summary_hier.items():
    print(f'  {k:30s} {v}')


Accuracy by level (higher = coarser prediction):
  rechtsgebied_accuracy          0.829
  wetboek_accuracy               0.829
  boek_accuracy                  0.512
  titel_accuracy                 0.488
  afdeling_accuracy              0.488
  article_id_accuracy            0.378


In [25]:
# Per-atom breakdown: where did the prediction land in the tree?
eval_hier[['atom_id',
           'rechtsgebied_true', 'rechtsgebied_pred', 'rechtsgebied_hit',
           'boek_true',         'boek_pred',         'boek_hit',
           'article_id_true',   'article_id_pred',   'article_id_hit']]


,atom_id,rechtsgebied_true,rechtsgebied_pred,rechtsgebied_hit,boek_true,boek_pred,boek_hit,article_id_true,article_id_pred,article_id_hit
0,6:162(1).a,private_law,private_law,True,Boek 6,Boek 6,True,6:162,6:162,True
1,6:162(2).a,private_law,private_law,True,Boek 6,Boek 6,True,6:162,6:162,True
2,6:162(2).b,private_law,private_law,True,Boek 6,Boek 6,True,6:162,6:162,True
3,6:162(3).a,private_law,private_law,True,Boek 6,Boek 6,True,6:162,6:75,False
4,6:170(1).a,private_law,private_law,True,Boek 6,Boek 6,True,6:170,6:170,True
...,...,...,...,...,...,...,...,...,...,...
77,synth:14(1).a,synthetic,synthetic,True,NaN,NaN,False,synth:14,synth:2,False
78,synth:14(1).b,synthetic,synthetic,True,NaN,NaN,False,synth:14,synth:13,False
79,synth:15(1).a,synthetic,synthetic,True,NaN,NaN,False,synth:15,synth:3,False
80,synth:15(1).b,synthetic,synthetic,True,NaN,NaN,False,synth:15,synth:15,True


## 9. Chained reasoning demo

The classifier answers "which atom is most similar to my query?" A query about
medical records finds 7:455, and stops there.

But real legal reasoning doesn't stop at one article. If a query lands on the
destruction duty (7:455), the reader also needs the retention override that
depends on it (7:454(3).a "Without prejudice to article 455..."). And often
the peripheral articles that share concepts.

This section walks the graph from the anchor match through three edge channels:

- **Reference edges** : explicit statutory citations (from `explicit_references`)
- **Shared-tag edges** : atoms overlapping on annotation tags (IDF-weighted)
- **Classifier ranking** : the anchor itself

Every step in the chain records *how* it was reached, so the result is
auditable.

### 9.1 Build the inputs

In [26]:
from src.data.reasoning import chained_reasoning, explain_chain

# Reference edges from section 8.4
if 'ref_edges' not in dir():
    ref_edges, _ = build_reference_edges(df_tax, return_unresolved=True)

# Weighted edges from section 5
if 'edges_weighted' not in dir():
    edges_weighted = build_edge_table(df, weights=weights_idf)

print(f'{len(df)} atoms, {len(edges_weighted)} tag edges, {len(ref_edges)} reference edges')

82 atoms, 639 tag edges, 5 reference edges


### 9.2 Query 1 Destruction chain

A paraphrase of "can a patient request their file be destroyed?" Should anchor
on 7:455 and then follow the reference edge back to 7:454(3).a, the retention
override. If the graph is doing real work, both articles surface.

In [27]:
destruction_query = {
    'atom_id': 'Q_destruction',
    'article_id': 'UNKNOWN',
    'actors': ['patient', 'care provider'],
    'acts': ['destroys'],
    'residual': ['file', 'request'],
    'temporal': ['after request'],
    'legal_relations': [], 'explicit_references': [], 'hierarchies': [], 'geographical_domain': [],
}

result_destruction = chained_reasoning(
    destruction_query, df, ref_edges, edges_weighted,
    weights=weights_idf, sim_pairs=sim_pairs,
    alpha=0.5, top_k=2, max_hops=1,
    min_edge_strength=2.0, max_expansions_per_atom=3,
)
print(explain_chain(result_destruction))

Query: Q_destruction
  tags: actors=['patient', 'care provider'] · acts=['destroys'] · residual=['file', 'request']

Anchor matches (hop 0, classifier):
  7:455(1).a      via: classifier rank 1  [classifier, score=7.63]
                  → The care provider shall destroy the data from the file after a written or electronic request to that effect b…
  7:454(2).a      via: classifier rank 2  [classifier, score=4.27]
                  → The care provider shall, upon request, add to the file a statement issued by the patient

Hop 1 — graph expansion:
  7:457(1).a      via: 7:457(1).a references 7:454 ('article 454')  [reference_in]
                  → Without prejudice to article 448 paragraph 3, second sentence, the care provider ensures that no information…
  7:448(1).b      via: shares ['care provider', 'patient', 'treatment agreement', 'upon request'] with 7:454(2).a  [shared_tags, score=7.41]
                  → The care provider shall inform the patient in writing upon request
  7:44

### 9.3 Query 2 Breach-of-duty chain

A paraphrase of "the care provider failed to keep proper records, causing harm
to the patient." Should cross-domain of WGBO's care-provider language and Book
6's obligation language share enough concepts that the chain surfaces both.

Notice there is no explicit reference edge between WGBO and Book 6 in the
current corpus. So this chain relies entirely on shared-tag edges . It is a test of whether the tag overlap is picking up real legal similarity.

In [28]:
breach_query = {
    'atom_id': 'Q_breach',
    'article_id': 'UNKNOWN',
    'actors': ['care provider', 'patient'],
    'acts': ['compensates'],
    'legal_relations': ['obligation'],
    'residual': ['damage', 'failure to perform', 'file'],
    'temporal': [], 'explicit_references': [], 'hierarchies': [], 'geographical_domain': [],
}

result_breach = chained_reasoning(
    breach_query, df, ref_edges, edges_weighted,
    weights=weights_idf, sim_pairs=sim_pairs,
    alpha=0.5, top_k=3, max_hops=1,
    min_edge_strength=2.0, max_expansions_per_atom=3,
)
print(explain_chain(result_breach))

Query: Q_breach
  tags: actors=['care provider', 'patient'] · acts=['compensates'] · legal_relations=['obligation'] · residual=['damage', 'failure to perform', 'file']

Anchor matches (hop 0, classifier):
  6:74(1).a       via: classifier rank 1  [classifier, score=6.35]
                  → Every failure to perform an obligation obliges the debtor to compensate the damage that the creditor suffers…
  6:162(1).a      via: classifier rank 2  [classifier, score=3.18]
                  → One who commits an unlawful act against another, which can be attributed to him, is obliged to compensate the…
  7:455(1).a      via: classifier rank 3  [classifier, score=2.39]
                  → The care provider shall destroy the data from the file after a written or electronic request to that effect b…

Hop 1 — graph expansion:
  6:76(1).a       via: shares ['debtor', 'obligation', 'performance'] with 6:74(1).a  [shared_tags, score=8.57]
                  → If a debtor uses the assistance of other per

### 9.4 Query 3 Hospital Liability chain

A paraphrase of "the hospital is being sued for a doctor's mistake." Should
anchor on 6:170 (vicarious liability for subordinates) and expand to the
related tort and contract articles that share concepts like `damage`, `fault`,
`liability`.

In [29]:
hospital_query = {
    'atom_id': 'Q_hospital',
    'article_id': 'UNKNOWN',
    'actors': ['employer', 'subordinate', 'third party'],
    'residual': ['damage', 'fault', 'liability'],
    'legal_relations': ['service relationship'],
    'acts': [], 'temporal': [], 'explicit_references': [], 'hierarchies': [], 'geographical_domain': [],
}

result_hospital = chained_reasoning(
    hospital_query, df, ref_edges, edges_weighted,
    weights=weights_idf, sim_pairs=sim_pairs,
    alpha=0.5, top_k=2, max_hops=1,
    min_edge_strength=2.0, max_expansions_per_atom=3,
)
print(explain_chain(result_hospital))

Query: Q_hospital
  tags: actors=['employer', 'subordinate', 'third party'] · legal_relations=['service relationship'] · residual=['damage', 'fault', 'liability']

Anchor matches (hop 0, classifier):
  6:170(1).a      via: classifier rank 1  [classifier, score=11.05]
                  → For damage caused to a third party by a fault of a subordinate, the person in whose service the subordinate p…
  6:170(1).c      via: classifier rank 2  [classifier, score=6.01]
                  → and the person in whose service he was had, by virtue of their relevant legal relationship, control over the…

Hop 1 — graph expansion:
  6:76(1).a       via: shares ['conduct'] with 6:170(1).c  [shared_tags, score=3.32]
                  → If a debtor uses the assistance of other persons in performing an obligation, he is liable for their conduct…
  6:162(3).a      via: shares ['fault'] with 6:170(1).c  [shared_tags, score=2.63]
                  → An unlawful act can be attributed to the perpetrator if it i

### 9.5 Structured result for the UI

`chained_reasoning()` returns a structured dict, not just text. This is what a
Streamlit UI would consume: `articles_touched` for a highlighted subgraph,
`chain` for the "explanation" panel, `primary_matches` for the ranked-answers
list.

In [30]:
print('Articles surfaced by the destruction query:')
print(' ', result_destruction['articles_touched'])
print()
print('Full chain (DataFrame view):')
import pandas as pd
pd.DataFrame(result_destruction['chain'])[
    ['hop', 'atom_id', 'article_id', 'edge_kind', 'via']
]

Articles surfaced by the destruction query:
  ['7:446', '7:448', '7:454', '7:455', '7:457']

Full chain (DataFrame view):


,hop,atom_id,article_id,edge_kind,via
0,0,7:455(1).a,7:455,classifier,classifier rank 1
1,0,7:454(2).a,7:454,classifier,classifier rank 2
2,1,7:457(1).a,7:457,reference_in,7:457(1).a references 7:454 ('article 454')
3,1,7:448(1).b,7:448,shared_tags,"shares ['care provider', 'patient', 'treatment agreement', 'upon r..."
4,1,7:446(1).a,7:446,shared_tags,"shares ['care provider', 'patient', 'treatment agreement'] with 7:..."
5,1,7:454(3).a,7:454,reference_in,7:454(3).a references 7:455 ('article 455')
6,1,7:454(1).b,7:454,shared_tags,"shares ['care provider', 'patient', 'data', 'file'] with 7:455(1).a"
7,1,7:454(1).d,7:454,shared_tags,"shares ['care provider', 'data', 'file'] with 7:455(1).a"
